# COGS 109 — Project Summary
## How Hyperscale Data Centers Affect Local Communities

A one-page map of the project for the human writers. It states the headline findings and points to
the two detailed notebooks. **These notebooks explain what was done and found; they do not write the
paper.**

---

### The two notebooks
1. **`data_overview.ipynb`** — *Data collection.* How the 45 treated + 124 control counties were
   chosen, and how the 7 federal / COVID-era data sources were pulled and merged into the national
   panel (169 counties × 2010–2024).
2. **`analysis.ipynb`** — *Analysis.* The difference-in-differences results, the COVID-controlled
   specification, the event study (pre-trends), and the selection classifier.

### Supersedes the old 3-state version
An earlier draft studied only 3 Meta counties in OH/VA/NE and reported a **significant** +0.89 ¢/kWh
electricity effect. That design had only 3 treated counties and could not separate data-center siting
from the treatment itself. This national version (45 treated counties, matched controls,
selection-aware) is what these notebooks now describe — and it tells a more careful story.

### The headline findings

- **Electricity rate:** the national effect attenuates to **~+0.15 ¢/kWh and is not statistically
  distinguishable from zero** (the old 3-state estimate was +0.89 and significant).
- **COVID is not the explanation:** adding explicit COVID controls (deaths + work-from-home) to the
  model changes the electricity coefficient by only **~10%** on identical data. The pandemic is not
  driving the result.
- **Pre-trends are flat:** the event study shows treated and control counties on parallel paths
  before opening, so the difference-in-differences design is defensible.
- **Selection is weak within the matched design:** a classifier using only pre-treatment county
  characteristics barely beats chance (AUC ≈ 0.59) — partly because controls were deliberately matched
  to treated counties. Identification therefore rests on the within-county DiD comparison.

The honest takeaway: **the suggestive 3-county signal does not survive at national scale, and it is
explicitly not a COVID artifact.** That is a more defensible and more interesting result than the
original.

In [1]:
# Headline numbers, loaded live from the results so this page can't drift out of sync
import pandas as pd
ct = pd.read_csv('results_national/comparison_table.csv')
cc = pd.read_csv('results_national/classifier_comparison.csv')
e  = ct[ct.outcome=='elec_rate_cents_kwh'].set_index('model')
m3, m3c = e.loc['M3'], e.loc['M3+C']

print('ELECTRICITY (national DiD):')
print(f'  M3   dc_active = {m3.dc_active_coef_ols:+.3f} c/kWh  CI [{m3.dc_active_ci_lo_ols:.3f}, {m3.dc_active_ci_hi_ols:.3f}]  (spans 0)')
print(f'  M3+C dc_active = {m3c.dc_active_coef_ols:+.3f} c/kWh  '
      f'(change {100*(m3c.dc_active_coef_ols-m3.dc_active_coef_ols)/abs(m3.dc_active_coef_ols):+.0f}% when COVID controls added)')
print('\nSELECTION CLASSIFIER:')
best = cc[cc.model.isin(['Logistic','KNN'])].sort_values('cv_auc').iloc[-1]
print(f'  best model {best.model} ({best.params}): CV AUC = {best.cv_auc:.3f}  (chance = 0.5)')
print(f'  accuracy {best.cv_accuracy:.3f} vs base-rate baseline {cc.iloc[-1].cv_accuracy:.3f}')
print('\nRead the two notebooks above for the full story.')

ELECTRICITY (national DiD):
  M3   dc_active = +0.154 c/kWh  CI [-0.268, 0.576]  (spans 0)
  M3+C dc_active = +0.138 c/kWh  (change -10% when COVID controls added)

SELECTION CLASSIFIER:
  best model KNN (K=10): CV AUC = 0.592  (chance = 0.5)
  accuracy 0.775 vs base-rate baseline 0.769

Read the two notebooks above for the full story.
